# **Fama–French 5-Factor Model**
### **Factors:** `MKT` | `SMB` | `HML` | `RMW` | `CMA`

---

### **Overview**
- The **Fama–French 5-factor model** explains stock returns using **five systematic risk factors**
- It extends:
  - **CAPM**
  - **Fama–French 3-factor model**
- Adds:
  - **Profitability**
  - **Investment behavior**

---

### **Model Equation**
- **`Ri - Rf = α + βm(MKT) + βs(SMB) + βh(HML) + βr(RMW) + βc(CMA) + ε`**

---

### **Terms Explanation**
- **`Ri`** → Return of asset
- **`Rf`** → Risk-free rate
- **`MKT`** → Market return minus risk-free rate
- **`α` (Alpha)** → Excess return
- **`ε`** → Error term

---

### **Factor Definitions**

#### **1. `MKT` (Market Risk Premium)**
- **`MKT = Rm - Rf`**
- Captures overall market movement

---

#### **2. `SMB` (Small Minus Big)**
- Small-cap returns − Large-cap returns
- Proxy: Market capitalization

---

#### **3. `HML` (High Minus Low)**
- Value stocks − Growth stocks
- Proxy: Book-to-market ratio

---

#### **4. `RMW` (Robust Minus Weak)**
- High profitability − Low profitability
- Proxy: ROE, ROA

---

#### **5. `CMA` (Conservative Minus Aggressive)**
- Low investment − High investment
- Proxy: Asset growth

---

### **Intuition**
- Returns are driven by **systematic factor exposure**
- Each stock has **factor sensitivities (`β`)**
- These determine expected return

---

### **Why This Model Matters**
- Explains cross-sectional return differences
- Used in:
  - Portfolio construction
  - Quant investing
  - Asset pricing

---

### **Limitations**
- Linear assumptions
- Static factors
- Missing:
  - `Momentum`
  - Nonlinear effects

---

### **Project Connection**
- Construct all 5 factors from raw data
- Use them to:
  - Explain returns
  - Generate alpha
- Extend with ML

---

### **Expected Outputs**
- Factor series: `MKT`, `SMB`, `HML`, `RMW`, `CMA`
- Factor loadings
- Alpha
- Backtest results

### Phase 1: Data ingestion and Cleaning (Nifty 500: From 01-01-2011 till date)

In [1]:
# ==========================================
# Phase 1: Data Collection (NIFTY 100)
# ==========================================

from nselib import capital_market
import pandas as pd
from datetime import date, datetime
import time
import os
import requests
import feedparser
from urllib.parse import quote_plus
import shutil
from pathlib import Path
from transformers import pipeline


C:\Users\jayan\anaconda3\envs\ml_alpha_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# import sys
# !{sys.executable} -m pip install nselib
# !{sys.executable} -m pip install feedparser

In [3]:
# =====================================
# STEP 0: Project Setup
# =====================================

PROJECT_ROOT = Path().resolve().parent
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"

# Clean full data directory
if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)

RAW_DIR.mkdir(parents=True, exist_ok=True)

print("Clean data directories ready.")


Clean data directories ready.


In [4]:
FIN_PROCESSED_DIR = DATA_DIR / "fin_data_processed"
FIN_BACKUPS_DIR = DATA_DIR / "fin_data_backups"

FIN_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIN_BACKUPS_DIR.mkdir(parents=True, exist_ok=True)


In [5]:
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [6]:
# ====================================
# STEP 1: Fetch NIFTY 100 tickers
# ====================================

url = "https://archives.nseindia.com/content/indices/ind_nifty500list.csv"
df_list = pd.read_csv(url)

tickers = df_list["Symbol"].tolist()

print(f"Total tickers fetched: {len(tickers)}")


Total tickers fetched: 501


In [7]:
df_list.head()

,Company Name,Industry,Symbol,Series,ISIN Code
0,360 ONE WAM Ltd.,Financial Services,360ONE,EQ,INE466L01038
1,3M India Ltd.,Diversified,3MINDIA,EQ,INE470A01017
2,ABB India Ltd.,Capital Goods,ABB,EQ,INE117A01022
3,ACC Ltd.,Construction Materials,ACC,EQ,INE012A01025
4,ACME Solar Holdings Ltd.,Power,ACMESOLAR,EQ,INE622W01025


In [8]:
df_list.shape

(501, 5)

In [9]:
# ==================================
# STEP 2: Fetch Price + News
# ==================================

def fetch_all_data(symbols, start, end):

    failed = []

    for symbol in symbols:

        try:
            print(f"\n Fetching {symbol} ...")

            df = capital_market.price_volume_and_deliverable_position_data(
                symbol=symbol,
                from_date=start.strftime("%d-%m-%Y"),
                to_date=end.strftime("%d-%m-%Y"),
            )

            if df.empty:
                print(f" No price data for {symbol}")
                failed.append(symbol)
                continue

            df.columns = df.columns.str.lower().str.strip()

            # Keep only EQ series
            if "series" in df.columns:
                df = df[df["series"].str.upper() == "EQ"].copy()

                if df.empty:
                    print(f" {symbol} has no EQ data")
                    failed.append(symbol)
                    continue

            # Save price
            df.to_csv(RAW_DIR / f"{symbol}.csv", index=False)
            print(f" Price saved")

        except Exception as e:
            print(f"Failed: {symbol} | {e}")
            failed.append(symbol)

        time.sleep(1)

    print("\n Data Fetch Completed.")
    print(f" Success: {len(symbols) - len(failed)} | Failed: {len(failed)}")

    if failed:
        pd.DataFrame(failed, columns=["Symbol"]).to_csv(
            DATA_DIR / "failed_tickers.csv", index=False
        )
        print("Failed tickers saved.")

In [10]:
## =======================================================================
# STEP 3: Execute for retreiving stock price data of 500 stocks from 2011
# =======================================================================

START_DATE = date(2011, 1, 1)
END_DATE = date.today()

fetch_all_data(tickers, START_DATE, END_DATE)

print("Total RAW files:", len(list(RAW_DIR.glob("*.csv"))))


 Fetching 360ONE ...
 Price saved

 Fetching 3MINDIA ...
 Price saved

 Fetching ABB ...
 Price saved

 Fetching ACC ...
 Price saved

 Fetching ACMESOLAR ...
 Price saved

 Fetching AIAENG ...
 Price saved

 Fetching APLAPOLLO ...
 Price saved

 Fetching AUBANK ...
 Price saved

 Fetching AWL ...
 Price saved

 Fetching AADHARHFC ...
 Price saved

 Fetching AARTIIND ...
 Price saved

 Fetching AAVAS ...
 Price saved

 Fetching ABBOTINDIA ...
 Price saved

 Fetching ACE ...
 Price saved

 Fetching ACUTAAS ...
 Price saved

 Fetching ADANIENSOL ...
 Price saved

 Fetching ADANIENT ...
 Price saved

 Fetching ADANIGREEN ...
 Price saved

 Fetching ADANIPORTS ...
 Price saved

 Fetching ADANIPOWER ...
 Price saved

 Fetching ATGL ...
 Price saved

 Fetching ABCAPITAL ...
 Price saved

 Fetching ABFRL ...
 Price saved

 Fetching ABLBL ...
 Price saved

 Fetching ABREL ...
 Price saved

 Fetching ABSLAMC ...
 Price saved

 Fetching CPPLUS ...
 Price saved

 Fetching AEGISLOG ...
 Price sav

 Price saved

 Fetching IDFCFIRSTB ...
 Price saved

 Fetching IFCI ...
 Price saved

 Fetching IIFL ...
 Price saved

 Fetching IRB ...
 Price saved

 Fetching IRCON ...
 Price saved

 Fetching ITCHOTELS ...
 Price saved

 Fetching ITC ...
 Price saved

 Fetching ITI ...
 Price saved

 Fetching INDGN ...
 Price saved

 Fetching INDIACEM ...
 Price saved

 Fetching INDIAMART ...
 Price saved

 Fetching INDIANB ...
 Price saved

 Fetching IEX ...
 Price saved

 Fetching INDHOTEL ...
 Price saved

 Fetching IOC ...
 Price saved

 Fetching IOB ...
 Price saved

 Fetching IRCTC ...
 Price saved

 Fetching IRFC ...
 Price saved

 Fetching IREDA ...
 Price saved

 Fetching IGL ...
 Price saved

 Fetching INDUSTOWER ...
 Price saved

 Fetching INDUSINDBK ...
 Price saved

 Fetching NAUKRI ...
 Price saved

 Fetching INFY ...
 Price saved

 Fetching INOXWIND ...
 Price saved

 Fetching INTELLECT ...
 Price saved

 Fetching INDIGO ...
 Price saved

 Fetching IGIL ...
 Price saved

 Fetching IKS


 Fetching TENNIND ...
 Price saved

 Fetching NIACL ...
 Price saved

 Fetching RAMCOCEM ...
 Price saved

 Fetching THERMAX ...
 Price saved

 Fetching TIMKEN ...
 Price saved

 Fetching TITAGARH ...
 Price saved

 Fetching TITAN ...
 Price saved

 Fetching TORNTPHARM ...
 Price saved

 Fetching TORNTPOWER ...
 Price saved

 Fetching TARIL ...
 Price saved

 Fetching TRAVELFOOD ...
 Price saved

 Fetching TRENT ...
 Price saved

 Fetching TRIDENT ...
 Price saved

 Fetching TRITURBINE ...
 Price saved

 Fetching TIINDIA ...
 Price saved

 Fetching UCOBANK ...
 Price saved

 Fetching UNOMINDA ...
 Price saved

 Fetching UPL ...
 Price saved

 Fetching UTIAMC ...
 Price saved

 Fetching ULTRACEMCO ...
 Price saved

 Fetching UNIONBANK ...
 Price saved

 Fetching UBL ...
 Price saved

 Fetching UNITDSPR ...
 Price saved

 Fetching URBANCO ...
 Price saved

 Fetching USHAMART ...
 Price saved

 Fetching VTL ...
 Price saved

 Fetching VBL ...
 Price saved

 Fetching VEDL ...
 Price saved

In [11]:
# ======================================================
# STEP 4: Fetching Fundamental Data
# ======================================================

import warnings; warnings.filterwarnings("ignore")

folder_path = RAW_DIR
all_data = []

for file in os.listdir(folder_path):
    if file.endswith(".csv"):
        temp = pd.read_csv(os.path.join(folder_path, file), encoding='latin1')
        
        temp.columns = temp.columns.str.strip().str.lower()
        temp['symbol'] = file.replace(".csv", "")
        
        # Clean price
        temp['closeprice'] = temp['closeprice'].astype(str).str.replace(',', '')
        temp['closeprice'] = pd.to_numeric(temp['closeprice'], errors='coerce')
        
        # FIXED DATE PARSING
        temp['date'] = pd.to_datetime(temp['date'], dayfirst=True, errors='coerce')
        
        # Drop invalid dates
        temp = temp.dropna(subset=['date'])
        
        temp = temp.sort_values('date')
        temp.set_index('date', inplace=True)
        
        # Monthly resample
        monthly = temp.resample('M').last().reset_index()
        monthly['return'] = monthly['closeprice'].pct_change()
        
        monthly['year'] = monthly['date'].dt.year
        monthly['month'] = monthly['date'].dt.month
        
        monthly['effective_year'] = monthly['year']
        monthly.loc[monthly['month'] < 7, 'effective_year'] -= 1
        
        all_data.append(monthly)

price_data = pd.concat(all_data, ignore_index=True)
price_data = price_data.sort_values('date').reset_index(drop=True)

In [12]:
price_data.sort_values('date')

,date,"ã¯â»â¿""symbol""",series,prevclose,openprice,highprice,lowprice,lastprice,closeprice,averageprice,totaltradedquantity,turnoverinrs,no.oftrades,deliverableqty,%dlyqttotradedqty,symbol,return,year,month,effective_year
0,2011-01-31,MPHASIS,EQ,658.7,657.4,678.9,637.5,678.0,674.65,655.11,399219.0,2.615327e+08,NaN,243120.0,60.9,MPHASIS,NaN,2011,1,2010
181,2011-01-31,BEML,EQ,887.70,880.00,880.00,857.05,869.00,870.10,869.22,25589.0,2.224246e+07,NaN,14648.0,57.24,BEML,NaN,2011,1,2010
182,2011-01-31,GODREJIND,EQ,172.0,170.0,175.0,167.45,174.35,173.40,171.8,257863.0,4.430039e+07,NaN,70844.0,27.47,GODREJIND,NaN,2011,1,2010
183,2011-01-31,BLUEDART,EQ,"1,037.05",981.00,"1,040.00",981.00,"1,040.00",1031.15,"1,027.57",156.0,1.603010e+05,NaN,76.0,48.72,BLUEDART,NaN,2011,1,2010
184,2011-01-31,SOBHA,EQ,259.95,258.0,263.8,252.0,262.0,261.20,257.9,47581.0,1.227092e+07,NaN,16687.0,35.07,SOBHA,NaN,2011,1,2010
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69622,2026-09-30,DEEPAKNTR,EQ,"1,645.80","1,634.00","1,635.10","1,583.20","1,596.00",1587.80,"1,598.92",323567.0,5.173572e+08,25941.0,162983.0,59.27,DEEPAKNTR,-0.092375,2026,9,2026
69621,2026-09-30,DEEPAKFERT,EQ,"1,335.60","1,335.60","1,341.30","1,316.00","1,323.00",1320.40,"1,324.73",147816.0,1.958166e+08,15699.0,66980.0,50.13,DEEPAKFERT,-0.061549,2026,9,2026
69620,2026-09-30,SUNPHARMA,EQ,"1,861.00","1,845.00","1,854.50","1,834.00","1,840.00",1840.00,"1,844.45",1445071.0,2.665355e+09,54700.0,896962.0,71.97,SUNPHARMA,-0.072954,2026,9,2026
69633,2026-09-30,DMART,EQ,"3,690.00","3,677.80","3,706.20","3,650.00","3,688.00",3688.00,"3,673.83",710188.0,2.609110e+09,39044.0,411932.0,64.45,DMART,-0.054359,2026,9,2026


In [13]:
# ==========================================================
# SAVE MONTHLY RETURNS
# ==========================================================

returns_df = price_data[
    ['date', 'symbol', 'return']
].copy()

returns_df.to_csv(
    PROCESSED_DIR / "monthly_returns.csv",
    index=False
)

print("Saved: monthly_returns.csv")

Saved: monthly_returns.csv


In [14]:
# Total number of monthly periods
total_periods = price_data['date'].nunique()

# Keep stocks with at least 70% data
threshold = 0.7 * total_periods

valid_stocks = (
    price_data.groupby('symbol')['closeprice']
    .count()
)

valid_stocks = valid_stocks[valid_stocks > threshold].index

price_data = price_data[price_data['symbol'].isin(valid_stocks)].copy()

In [15]:
median_vol = (
    price_data.groupby('symbol')['totaltradedquantity']
    .median()
)

liquid_stocks = median_vol[median_vol > 1e5].index


In [16]:
print(len(price_data))
print(len(liquid_stocks))

57793
245


In [17]:
price_data = price_data[price_data['symbol'].isin(liquid_stocks)].copy()

In [18]:
price_data['totaltradedquantity'] = pd.to_numeric(
    price_data['totaltradedquantity'],
    errors='coerce'
)

In [19]:
price_data['turnover'] = (
    price_data['closeprice'] * price_data['totaltradedquantity']
)

In [20]:
# =====================================================================
# STEP 5: Fetching Risk-free returns (91-day Govt. Treasury bills)
# =====================================================================

url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=INDIR3TIB01STM"

df = pd.read_csv(url)
df['DATE'] = pd.to_datetime(df['observation_date'])

df = df.set_index('DATE')
df.rename(columns={'INDIR3TIB01STM': 'Rf'}, inplace=True) #Rf means risk-free returns

df['Rf'] = df['Rf'] / 100
df = df.resample('D').ffill()
df['Rf_daily'] = df['Rf'] / 365

df = df[['Rf_daily']].copy()

df.to_csv(DATA_DIR / "rf_india.csv")

In [21]:
rf = pd.read_csv(DATA_DIR / "rf_india.csv", parse_dates=['DATE'])
rf = rf.set_index('DATE')

rf_monthly = rf.resample('M').apply(lambda x: (1 + x).prod() - 1)
rf_monthly = rf_monthly.rename(columns={'Rf_daily': 'Rf_monthly'})

In [22]:
rf_monthly.to_csv(
    PROCESSED_DIR / "rf_monthly.csv"
)

print("Saved: rf_monthly.csv")

Saved: rf_monthly.csv


In [23]:
# =====================================================================
# STEP 6: Financial Data Pipeline
# =====================================================================

In [24]:
import yfinance as yf
import requests
import pandas as pd
import numpy as np
import time, re, json
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


def create_session():
    s = requests.Session()
    s.mount("https://", HTTPAdapter(max_retries=Retry(
        total=3, backoff_factor=2,
        status_forcelist=[429, 500, 502, 503, 504])))
    s.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Accept": "application/json, */*",
    })
    return s


In [25]:
symbols = tickers

In [26]:
# =============================================================================
# SOURCE 1 — yfinance with the correct API calls (not .quarterly_financials)
# Gets ~2015 onwards for most Nifty500 stocks
# =============================================================================

def fetch_yfinance(symbol):
    """
    Uses get_income_stmt / get_balance_sheet instead of the
    .quarterly_financials property — these hit a different Yahoo endpoint
    that returns more history (typically 12-16 quarters vs 4-5).
    """
    try:
        ticker = yf.Ticker(f"{symbol}.NS")

        # These methods return more history than the .quarterly_financials property
        inc = ticker.get_income_stmt(freq="quarterly", as_dict=False)
        bal = ticker.get_balance_sheet(freq="quarterly", as_dict=False)

        if inc is None or inc.empty:
            # Final fallback to the property
            inc = ticker.quarterly_financials
        if bal is None or bal.empty:
            bal = ticker.quarterly_balance_sheet

        if inc is None or inc.empty or bal is None or bal.empty:
            return None, "yf_empty"

        NI_KEYS = [
            "Net Income",
            "Net Income From Continuing Operations",
            "Net Income Including Noncontrolling Interests",
            "Net Income Common Stockholders",
        ]
        TA_KEYS = ["Total Assets"]

        ni_row = next((inc.loc[k] for k in NI_KEYS if k in inc.index), None)
        ta_row = next((bal.loc[k] for k in TA_KEYS if k in bal.index), None)

        if ni_row is None: return None, "yf_no_net_income"
        if ta_row is None: return None, "yf_no_assets"

        all_dates = sorted(
            set(ni_row.dropna().index) | set(ta_row.dropna().index)
        )

        records = []
        for dt in all_dates:
            month = dt.month
            if   month in [4,5,6]:   qtr, fy = "Q1", dt.year + 1
            elif month in [7,8,9]:   qtr, fy = "Q2", dt.year + 1
            elif month in [10,11,12]: qtr, fy = "Q3", dt.year + 1
            else:                     qtr, fy = "Q4", dt.year

            ni_val = ni_row.get(dt)
            ta_val = ta_row.get(dt)

            if pd.isna(ni_val) and pd.isna(ta_val):
                continue

            records.append({
                "year":         fy,
                "quarter":      qtr,
                "net_income":   float(ni_val) / 1e7 if pd.notna(ni_val) else None,
                "total_assets": float(ta_val) / 1e7 if pd.notna(ta_val) else None,
                "source":       "yfinance",
            })

        if not records:
            return None, "yf_no_records"

        df = (pd.DataFrame(records)
              .drop_duplicates(subset=["year", "quarter"])
              .sort_values(["year", "quarter"])
              .reset_index(drop=True))
        return df, None

    except Exception as e:
        return None, f"yf_error: {e}"


In [28]:
# =============================================================================
# SOURCE 2 — Screener.in annual data (HTML table scrape, NOT the JSON API)
# Gets 2014-2015 onwards reliably, annual only
# =============================================================================

from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import re
import time


def fetch_screener_html(symbol, session):
    """
    Scrapes annual P&L and Balance Sheet from Screener.in HTML.

    Extracted annual variables:
        P&L:
            - sales
            - operating_profit
            - other_income
            - interest_expense
            - depreciation
            - net_income

        Balance Sheet:
            - total_assets
            - book_value

    Operating Profitability (FF5 / RMW) is subsequently calculated as:

        OP = (Operating Profit - Interest Expense) / Book Equity

    This corresponds to the Fama-French profitability construction
    using operating income before depreciation/amortization minus
    interest expense, scaled by book equity.

    IMPORTANT:
    The accounting observation is the fiscal-year observation. The
    Phase 2 formation logic should subsequently use the LAST fiscal
    year ending in t-1 for portfolios formed in June of year t.
    """

    urls = [
        f"https://www.screener.in/company/{symbol}/consolidated/?view=15years",
        f"https://www.screener.in/company/{symbol}/?view=15years",
    ]

    # ------------------------------------------------------------------ #
    # FIXED parse_table: first non-empty occurrence of each label wins
    # ------------------------------------------------------------------ #
    def parse_table(tbl):

        rows = {}
        headers = []

        EMPTY_VALS = {
            '',
            '-',
            'na',
            'n/a',
            'nan',
            '--'
        }

        for i, row in enumerate(tbl.find_all("tr")):

            cells = [
                c.get_text(strip=True)
                for c in row.find_all(["th", "td"])
            ]

            if not cells:
                continue

            # First row = headers
            if i == 0:
                headers = cells
                continue

            label = cells[0].strip()
            values = cells[1:]

            if not label:
                continue

            def row_has_data(vals):
                return any(
                    v.strip().lower() not in EMPTY_VALS
                    for v in vals
                )

            if label not in rows:

                rows[label] = values

            else:

                # Keep first real data row.
                # Replace only if previous row was empty
                # and new row contains actual data.
                if (
                    not row_has_data(rows[label])
                    and row_has_data(values)
                ):
                    rows[label] = values

        return headers, rows

    # ------------------------------------------------------------------ #
    # Safe value extractor
    # ------------------------------------------------------------------ #
    def get_val(row_vals, col_idx):

        try:

            v = (
                str(row_vals[col_idx])
                .replace(",", "")
                .replace("(", "-")
                .replace(")", "")
                .strip()
            )

            if v.lower() in {
                '',
                '-',
                'na',
                'n/a',
                'nan',
                '--',
                'ttm'
            }:
                return None

            return float(v)

        except (
            IndexError,
            ValueError,
            TypeError
        ):
            return None

    # ------------------------------------------------------------------ #
    # Try Screener URLs
    # ------------------------------------------------------------------ #
    for url in urls:

        try:

            res = session.get(
                url,
                timeout=15
            )

            if res.status_code != 200:
                continue

            if (
                len(res.text) < 5000
                or "No data" in res.text
            ):
                continue

            soup = BeautifulSoup(
                res.text,
                "html.parser"
            )

            sections = soup.find_all("section")

            pnl_table = None
            bal_table = None

            # ---------------------------------------------------------- #
            # Locate P&L and Balance Sheet tables
            # ---------------------------------------------------------- #
            for sec in sections:

                heading = (
                    sec.find("h2")
                    or sec.find("h3")
                )

                if not heading:
                    continue

                h = heading.text.lower().strip()

                table = sec.find("table")

                if not table:
                    continue

                if (
                    (
                        "profit" in h
                        or "income" in h
                        or "p&l" in h
                    )
                    and pnl_table is None
                ):

                    pnl_table = table

                elif (
                    "balance" in h
                    and bal_table is None
                ):

                    bal_table = table

            if (
                pnl_table is None
                or bal_table is None
            ):
                continue

            # ---------------------------------------------------------- #
            # Parse tables
            # ---------------------------------------------------------- #
            pnl_headers, pnl_rows = parse_table(
                pnl_table
            )

            bal_headers, bal_rows = parse_table(
                bal_table
            )

            # ---------------------------------------------------------- #
            # Year-to-column-index maps
            # ---------------------------------------------------------- #
            def build_year_index(headers):

                result = []

                for idx, h in enumerate(
                    headers[1:],
                    start=0
                ):

                    m = re.search(
                        r'(20\d{2})',
                        h
                    )

                    if m:
                        result.append(
                            (
                                idx,
                                m.group(1)
                            )
                        )

                return result

            pnl_year_indices = build_year_index(
                pnl_headers
            )

            bal_year_indices = build_year_index(
                bal_headers
            )

            if not pnl_year_indices:
                continue

            # Balance sheet year → column index
            bal_year_map = {
                yr: idx
                for idx, yr
                in bal_year_indices
            }

            # ========================================================== #
            # P&L VARIABLES
            # ========================================================== #

            # Sales / Revenue
            SALES_RE = re.compile(
                r"^sales$|^revenue$|income from operations",
                re.I
            )

            sales_vals_raw = None

            for label, vals in pnl_rows.items():

                if SALES_RE.search(label.strip()):
                    sales_vals_raw = vals
                    break

            # Operating Profit
            OP_RE = re.compile(
                r"^operating profit$",
                re.I
            )

            operating_profit_vals_raw = None

            for label, vals in pnl_rows.items():

                if OP_RE.search(label.strip()):
                    operating_profit_vals_raw = vals
                    break

            # Other Income
            OTHER_INCOME_RE = re.compile(
                r"^other income$",
                re.I
            )

            other_income_vals_raw = None

            for label, vals in pnl_rows.items():

                if OTHER_INCOME_RE.search(label.strip()):
                    other_income_vals_raw = vals
                    break

            # Interest Expense
            INTEREST_RE = re.compile(
                r"^interest$|interest expense",
                re.I
            )

            interest_vals_raw = None

            for label, vals in pnl_rows.items():

                if INTEREST_RE.search(label.strip()):
                    interest_vals_raw = vals
                    break

            # Depreciation
            DEPRECIATION_RE = re.compile(
                r"^depreciation$|depreciation & amortisation|"
                r"depreciation and amortisation",
                re.I
            )

            depreciation_vals_raw = None

            for label, vals in pnl_rows.items():

                if DEPRECIATION_RE.search(label.strip()):
                    depreciation_vals_raw = vals
                    break

            # Net Income
            NI_RE = re.compile(
                r"net profit|profit after tax|\bpat\b",
                re.I
            )

            ni_vals_raw = None

            for label, vals in pnl_rows.items():

                if NI_RE.search(label):
                    ni_vals_raw = vals
                    break

            # ========================================================== #
            # BALANCE SHEET VARIABLES
            # ========================================================== #

            # Total Assets
            ta_vals_raw = None

            for label, vals in bal_rows.items():

                if "total assets" in label.lower().strip():

                    ta_vals_raw = vals
                    break

            # Book Value / Equity
            #
            # Priority:
            #   1. Direct equity
            #   2. Share Capital + Reserves
            
            DIRECT_EQUITY_RE = re.compile(
                r"total equity|shareholders['\s]*equity|net worth|"
                r"stockholders['\s]*equity|total networth|"
                r"equity \(net worth\)",
                re.I
            )

            SC_RE = re.compile(
                r"equity capital|equity share capital|share capital",
                re.I
            )

            RES_RE = re.compile(
                r"\breserves\b|other equity|surplus",
                re.I
            )

            direct_equity_raw = None
            share_capital_raw = None
            reserve_raw = None

            for label, vals in bal_rows.items():

                if (
                    direct_equity_raw is None
                    and DIRECT_EQUITY_RE.search(label)
                ):
                    direct_equity_raw = vals

                if (
                    share_capital_raw is None
                    and SC_RE.search(label)
                ):
                    share_capital_raw = vals

                if (
                    reserve_raw is None
                    and RES_RE.search(label)
                ):
                    reserve_raw = vals

            # ========================================================== #
            # BUILD ANNUAL RECORDS
            # ========================================================== #

            records = []

            for pnl_col_idx, yr in pnl_year_indices:

                # P&L values
                sales = (
                    get_val(
                        sales_vals_raw,
                        pnl_col_idx
                    )
                    if sales_vals_raw is not None
                    else None
                )

                operating_profit = (
                    get_val(
                        operating_profit_vals_raw,
                        pnl_col_idx
                    )
                    if operating_profit_vals_raw is not None
                    else None
                )

                other_income = (
                    get_val(
                        other_income_vals_raw,
                        pnl_col_idx
                    )
                    if other_income_vals_raw is not None
                    else None
                )

                interest_expense = (
                    get_val(
                        interest_vals_raw,
                        pnl_col_idx
                    )
                    if interest_vals_raw is not None
                    else None
                )

                depreciation = (
                    get_val(
                        depreciation_vals_raw,
                        pnl_col_idx
                    )
                    if depreciation_vals_raw is not None
                    else None
                )

                net_income = (
                    get_val(
                        ni_vals_raw,
                        pnl_col_idx
                    )
                    if ni_vals_raw is not None
                    else None
                )

                # Balance Sheet column
                bal_col_idx = bal_year_map.get(
                    yr,
                    pnl_col_idx
                )

                total_assets = (
                    get_val(
                        ta_vals_raw,
                        bal_col_idx
                    )
                    if ta_vals_raw is not None
                    else None
                )

                # Book Value
                book_value = None

                # Priority 1: direct equity
                if direct_equity_raw is not None:

                    book_value = get_val(
                        direct_equity_raw,
                        bal_col_idx
                    )

                # Priority 2: Share Capital + Reserves
                if (
                    book_value is None
                    and share_capital_raw is not None
                    and reserve_raw is not None
                ):

                    share_capital = get_val(
                        share_capital_raw,
                        bal_col_idx
                    )

                    reserves = get_val(
                        reserve_raw,
                        bal_col_idx
                    )

                    if (
                        share_capital is not None
                        and reserves is not None
                    ):

                        book_value = (
                            share_capital
                            + reserves
                        )

                # ------------------------------------------------------ #
                # FF5 Operating Profitability
                #
                # OP numerator: Operating Profit - Interest Expense
                # Denominator: Book Equity
                # ------------------------------------------------------ #

                operating_profitability = None

                if (
                    operating_profit is not None
                    and interest_expense is not None
                    and book_value is not None
                    and book_value > 0
                ):

                    operating_profitability = (operating_profit - interest_expense) / book_value

                # ------------------------------------------------------ #
                # Store record
                # ------------------------------------------------------ #

                records.append({

                    "year": int(yr),

                    "quarter": "FY",

                    # P&L
                    "sales": sales,
                    "operating_profit": operating_profit,
                    "other_income": other_income,
                    "interest_expense": interest_expense,
                    "depreciation": depreciation,
                    "net_income": net_income,

                    # Balance Sheet
                    "total_assets": total_assets,
                    "book_value": book_value,

                    # FF5 profitability
                    "operating_profitability":
                        operating_profitability,

                    "source":
                        "screener_html",
                })

            # ---------------------------------------------------------- #
            # Validate records
            # ---------------------------------------------------------- #

            if not records:
                continue

            df = (
                pd.DataFrame(records)
                .drop_duplicates("year")
                .sort_values("year")
                .reset_index(drop=True)
            )

            return df, None

        except Exception as e:

            print(
                f"[Screener ERROR] {symbol}: {e}"
            )

            continue

        time.sleep(1.5)

    return None, "screener_html_failed"


In [29]:
# =============================================================================
# SOURCE 3 — NSE India quarterly results (XBRL XML, publicly accessible)
# More reliable than BSE for unauthenticated requests
# Gets 2014 onwards quarterly for most stocks
# =============================================================================

def fetch_nse_quarterly(symbol, session):
    """
    NSE publishes quarterly result XMLs via their public archive.
    No authentication required. Endpoint is stable.
    """
    url = (
        f"https://www.nseindia.com/api/results-comparisons?"
        f"index=equities&symbol={symbol}"
    )
    try:
        # NSE requires these specific headers
        nse_session = requests.Session()
        nse_session.headers.update({
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
            "Accept": "application/json",
            "Referer": "https://www.nseindia.com/",
            "Accept-Language": "en-US,en;q=0.9",
        })
        # NSE requires a cookie — get it first with a homepage visit
        nse_session.get("https://www.nseindia.com", timeout=10)
        time.sleep(1)

        res = nse_session.get(url, timeout=15)
        if res.status_code != 200:
            return None, f"nse_status_{res.status_code}"

        data = res.json()
        rows = data.get("data", [])
        if not rows:
            return None, "nse_no_data"

        records = []
        for row in rows:
            period = row.get("period", "")          # e.g. "Q2 FY2023"
            ni  = row.get("net_profit_loss")
            ta  = row.get("total_assets")

            qm = re.search(r'(Q[1-4])\s*FY(\d{2,4})', period, re.I)
            if qm:
                qtr = qm.group(1).upper()
                fy  = int(qm.group(2))
                fy  = 2000 + fy if fy < 100 else fy
            else:
                continue

            try:
                records.append({
                    "year":         fy,
                    "quarter":      qtr,
                    "net_income":   float(str(ni).replace(",","")) if ni else None,
                    "total_assets": float(str(ta).replace(",","")) if ta else None,
                    "source":       "nse_api",
                })
            except (ValueError, TypeError):
                continue

        if not records:
            return None, "nse_no_parseable_rows"

        df = (pd.DataFrame(records)
              .drop_duplicates(subset=["year","quarter"])
              .sort_values(["year","quarter"])
              .reset_index(drop=True))
        return df, None

    except Exception as e:
        return None, f"nse_error: {e}"


In [30]:
# =============================================================================
# MASTER PIPELINE
# =============================================================================

def run_pipeline(symbol, session):
    results = []

    # Source 1: yfinance (~2015 onwards, quarterly)
    df, err = fetch_yfinance(symbol)
    if df is not None and len(df) > 0:
        df.insert(0, "symbol", symbol)
        results.append(df)
        print(f"  yf: {len(df)} qtrs ({df['year'].min()}-{df['year'].max()})", end="")

    # Source 2: Screener HTML (~2014 onwards, annual)
    df, err = fetch_screener_html(symbol, session)
    if df is not None and len(df) > 0:
        df.insert(0, "symbol", symbol)
        results.append(df)
        print(f"  scr: {len(df)} yrs", end="")

    # Source 3: NSE API (~2014 onwards, quarterly)
    df, err = fetch_nse_quarterly(symbol, session)
    if df is not None and len(df) > 0:
        df.insert(0, "symbol", symbol)
        results.append(df)
        print(f"  nse: {len(df)} qtrs", end="")

    if not results:
        return None, "all_sources_failed"

    # Merge all sources — quarterly beats annual for same period
    # Priority: yfinance > nse_api > screener_html
    combined = pd.concat(results, ignore_index=True)
    
    # --- Extract book_value separately BEFORE deduplication ---
    # Screener carries book_value; yfinance/NSE do not.
    # Deduplication (yfinance wins) would discard Screener rows and lose book_value.
    bv_df = (
        combined[combined["book_value"].notna()][["symbol", "year", "quarter", "book_value"]]
        .drop_duplicates(subset=["symbol", "year"])
        .copy()
    ) if "book_value" in combined.columns else pd.DataFrame()

    # Deduplicate financial rows by source priority (yfinance > nse_api > screener_html)
    priority = {"yfinance": 0, "nse_api": 1, "screener_html": 2}
    combined["_rank"] = combined["source"].map(priority).fillna(9)
    combined = (combined
        .sort_values("_rank")
        .drop_duplicates(subset=["symbol", "year", "quarter"], keep="first")
        .drop(columns="_rank")
        .sort_values(["year", "quarter"])
        .reset_index(drop=True))

    # --- Re-attach book_value after dedup ---
    if not bv_df.empty:
        if "book_value" in combined.columns:
            combined = combined.drop(columns=["book_value"])
        combined = combined.merge(
            bv_df[["symbol", "year", "book_value"]].drop_duplicates(subset=["symbol", "year"]),
            on=["symbol", "year"],
            how="left"
        )

    return combined, None

In [31]:
# =============================================================================
# EXECUTION
# =============================================================================

session = create_session()
all_data, failures = [], {}

for i, sym in enumerate(symbols):
    print(f"\n[{i+1}/{len(symbols)}] {sym}", end="  ")

    df, err = run_pipeline(sym, session)

    if df is not None:
        df["net_income"]   = pd.to_numeric(df["net_income"],   errors="coerce")
        df["total_assets"] = pd.to_numeric(df["total_assets"], errors="coerce")
        df = df[df["total_assets"] > 0].dropna(subset=["net_income"])

        if len(df) > 0:
            all_data.append(df)
            print(f"  → {len(df)} total rows")
        else:
            failures[sym] = "all_rows_invalid"
            print(f"  → all rows invalid after cleaning")
    else:
        failures[sym] = err
        print(f"  → FAILED: {err}")

    time.sleep(1.5 if (i+1) % 20 != 0 else 5.0)

    if (i+1) % 50 == 0 and all_data:
        pd.concat(all_data, ignore_index=True).to_csv(FIN_BACKUPS_DIR / f"fundamentals_backup_{i+1}.csv")
        print(f"\n  [backup saved at step {i+1}]")


final_df = pd.concat(all_data, ignore_index=True)

# Debug: confirm book_value presence
print("Columns in final_df:", final_df.columns.tolist())
print("book_value non-null count:", final_df['book_value'].notna().sum() if 'book_value' in final_df.columns else "COLUMN MISSING")
screener_rows = final_df[final_df['source'] == 'screener_html'] if 'source' in final_df.columns else pd.DataFrame()
print("Screener rows:", len(screener_rows))
if not screener_rows.empty and 'book_value' in screener_rows.columns:
    print("Screener book_value non-null:", screener_rows['book_value'].notna().sum())

if "book_value" not in final_df.columns:
    final_df["book_value"] = np.nan

# ==========================================================
# CLEAN BOOK VALUE PIPELINE
# ==========================================================

# Convert safely
final_df['book_value'] = pd.to_numeric(
    final_df['book_value'],
    errors='coerce'
)

# Keep only required columns
book_df = final_df[
    ['symbol', 'year', 'quarter', 'book_value']
].copy()

# After book_df is built, verify year alignment
print("book_df year sample:\n", book_df[['symbol','year','quarter','book_value']].head(10))

# Remove missing values
book_df = book_df.dropna(subset=['book_value'])

print("Initial non-null BV rows:", len(book_df))

# Keep annual values
book_df = book_df[
    book_df['quarter'].isin(['FY', 'Q4', 'Q3', 'Q2', 'Q1'])
]

print("Annual BV rows:", len(book_df))

# Create fiscal year-end date
book_df['date'] = pd.to_datetime(
    book_df['year'].astype(str) + "-03-31"
)

# Remove duplicates
book_df = (
    book_df
    .sort_values(['symbol', 'date'])
    .drop_duplicates(['symbol', 'date'], keep='last')
)

# Convert Cr → ₹
book_df['book_value'] = book_df['book_value'] * 1e7

# ==========================================================
# Monthly expansion
# ==========================================================

monthly_frames = []

for sym in book_df['symbol'].unique():

    temp = (
        book_df[book_df['symbol'] == sym]
        .sort_values('date')
        .set_index('date')[['book_value']]
        .resample('M')
        .ffill()
        .reset_index()
    )

    temp['symbol'] = sym

    monthly_frames.append(temp)

# Safe concat
if len(monthly_frames) > 0:

    book_monthly = pd.concat(
        monthly_frames,
        ignore_index=True
    )

else:
    print("WARNING: book_monthly is empty. Diagnostics:")
    print(f"  book_df shape: {book_df.shape}")
    print(f"  final_df book_value non-null: {final_df['book_value'].notna().sum()}")
    raise ValueError(
        "book_monthly is empty — check that Screener is returning book_value "
        "and that deduplication is not dropping all Screener rows."
    )
    
# Lag to avoid lookahead bias
book_monthly['book_value'] = (
    book_monthly
    .groupby('symbol')['book_value']
    .shift(3)
)

print(
    "Book monthly non-null:",
    book_monthly['book_value'].notna().sum()
)

final_df = final_df[final_df["year"].astype(int) >= 2011].reset_index(drop=True)
final_df = final_df.sort_values(["symbol","year","quarter"]).reset_index(drop=True)

pd.DataFrame(list(failures.items()), columns=["symbol","reason"]).to_csv(FIN_BACKUPS_DIR / "failures.csv", index=False)

print(f"\nSymbols     : {final_df['symbol'].nunique()}")
print(f"Total rows  : {len(final_df)}")
print(f"Year range  : {final_df['year'].min()} → {final_df['year'].max()}")
print(f"Avg rows/co : {len(final_df)/final_df['symbol'].nunique():.1f}")
print(f"Failures    : {len(failures)}")
print(f"\nSource breakdown:\n{final_df['source'].value_counts()}")


[1/501] 360ONE    scr: 9 yrs  → 9 total rows

[2/501] 3MINDIA    scr: 7 yrs  → 7 total rows

[3/501] ABB    scr: 4 yrs  → 4 total rows

[4/501] ACC    scr: 12 yrs  → 12 total rows

[5/501] ACMESOLAR    scr: 7 yrs  → 7 total rows

[6/501] AIAENG    scr: 12 yrs  → 12 total rows

[7/501] APLAPOLLO    scr: 12 yrs  → 12 total rows

[8/501] AUBANK    scr: 4 yrs  → 4 total rows

[9/501] AWL    scr: 8 yrs  → 8 total rows

[10/501] AADHARHFC    scr: 8 yrs  → 8 total rows

[11/501] AARTIIND    scr: 12 yrs  → 12 total rows

[12/501] AAVAS    scr: 7 yrs  → 7 total rows

[13/501] ABBOTINDIA    scr: 12 yrs  → 12 total rows

[14/501] ACE    scr: 12 yrs  → 12 total rows

[15/501] ACUTAAS    scr: 10 yrs  → 10 total rows

[16/501] ADANIENSOL    scr: 12 yrs  → 12 total rows

[17/501] ADANIENT    scr: 12 yrs  → 12 total rows

[18/501] ADANIGREEN    scr: 11 yrs  → 11 total rows

[19/501] ADANIPORTS    scr: 12 yrs  → 12 total rows

[20/501] ADANIPOWER    scr: 12 yrs  → 12 total rows

[21/501] ATGL    scr: 

[162/501] ESCORTS    scr: 12 yrs  → 12 total rows

[163/501] ETERNAL    scr: 9 yrs  → 9 total rows

[164/501] EXIDEIND    scr: 12 yrs  → 12 total rows

[165/501] NYKAA    scr: 9 yrs  → 9 total rows

[166/501] FEDERALBNK    scr: 12 yrs  → 12 total rows

[167/501] FACT    scr: 11 yrs  → 11 total rows

[168/501] FINCABLES    scr: 11 yrs  → 11 total rows

[169/501] FSL    scr: 12 yrs  → 12 total rows

[170/501] FIVESTAR    scr: 5 yrs  → 5 total rows

[171/501] FORCEMOT    scr: 12 yrs  → 12 total rows

[172/501] FORTIS    scr: 12 yrs  → 12 total rows

[173/501] GAIL    scr: 12 yrs  → 12 total rows

[174/501] GVT&D    scr: 1 yrs  → 1 total rows

[175/501] GMRAIRPORT    scr: 12 yrs  → 12 total rows

[176/501] GABRIEL    scr: 3 yrs  → 3 total rows

[177/501] GALLANTT    scr: 12 yrs  → 12 total rows

[178/501] GRSE    scr: 12 yrs  → 12 total rows

[179/501] GICRE    scr: 12 yrs  → 12 total rows

[180/501] GILLETTE    scr: 12 yrs  → 12 total rows

[181/501] GLAND    scr: 7 yrs  → 7 total rows

[

[323/501] MSUMI    scr: 6 yrs  → 6 total rows

[324/501] MOTILALOFS    scr: 12 yrs  → 12 total rows

[325/501] MPHASIS    scr: 12 yrs  → 12 total rows

[326/501] MCX    scr: 12 yrs  → 12 total rows

[327/501] MUTHOOTFIN    scr: 12 yrs  → 12 total rows

[328/501] NATCOPHARM    scr: 12 yrs  → 12 total rows

[329/501] NBCC    scr: 12 yrs  → 12 total rows

[330/501] NCC    scr: 12 yrs  → 12 total rows

[331/501] NHPC    scr: 12 yrs  → 12 total rows

[332/501] NLCINDIA    scr: 12 yrs  → 12 total rows

[333/501] NMDC    scr: 12 yrs  → 12 total rows

[334/501] NSLNISP    scr: 7 yrs  → 5 total rows

[335/501] NTPCGREEN    scr: 4 yrs  → 4 total rows

[336/501] NTPC    scr: 12 yrs  → 12 total rows

[337/501] NH    scr: 12 yrs  → 12 total rows

[338/501] NATIONALUM    scr: 12 yrs  → 12 total rows

[339/501] NAVA    scr: 12 yrs  → 12 total rows

[340/501] NAVINFLUOR    scr: 12 yrs  → 12 total rows

[341/501] NESTLEIND    scr: 4 yrs  → 3 total rows

[342/501] NETWEB    scr: 8 yrs  → 8 total rows

[


[484/501] VIJAYA    scr: 9 yrs  → 9 total rows

[485/501] VMM    scr: 7 yrs  → 7 total rows

[486/501] IDEA    scr: 12 yrs  → 12 total rows

[487/501] VOLTAS    scr: 12 yrs  → 12 total rows

[488/501] WAAREEENER    scr: 7 yrs  → 7 total rows

[489/501] WELCORP    scr: 12 yrs  → 12 total rows

[490/501] WELSPUNLIV    scr: 12 yrs  → 12 total rows

[491/501] WHIRLPOOL    scr: 9 yrs  → 9 total rows

[492/501] WIPRO    scr: 12 yrs  → 12 total rows

[493/501] WOCKPHARMA    scr: 12 yrs  → 12 total rows

[494/501] YESBANK    scr: 12 yrs  → 12 total rows

[495/501] ZFCVINDIA    scr: 5 yrs  → 5 total rows

[496/501] ZEEL    scr: 12 yrs  → 12 total rows

[497/501] ZENTEC    scr: 12 yrs  → 12 total rows

[498/501] ZENSARTECH    scr: 12 yrs  → 12 total rows

[499/501] ZYDUSLIFE    scr: 12 yrs  → 12 total rows

[500/501] ZYDUSWELL    scr: 12 yrs  → 12 total rows

  [backup saved at step 500]

[501/501] ECLERX    scr: 12 yrs  → 12 total rows
Columns in final_df: ['symbol', 'year', 'quarter', 'sales'

Check these dfs: final_df, fund_monthly, asset_growth_formation, price_data, market_cap_df, dec_me, book_equity_annual
before they form in subsequent cells.

In [32]:
# Got errors several times while fetching the data, so running diagnostics for a single company to check the tags

"""
Run this as a STANDALONE cell in your notebook BEFORE the main pipeline.
It will print the exact HTML structure Screener returns for one stock,
so we can see exactly what parse_table is working with.
"""

import requests
from bs4 import BeautifulSoup
import re

def diagnose_screener(symbol="RELIANCE"):
    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml",
        "Accept-Language": "en-US,en;q=0.9",
    })

    urls = [
        f"https://www.screener.in/company/{symbol}/consolidated/?view=10years",
        f"https://www.screener.in/company/{symbol}/?view=10years",
    ]

    for url in urls:
        print(f"\nFetching: {url}")
        res = session.get(url, timeout=15)
        print(f"Status: {res.status_code}  |  Length: {len(res.text)}")
        if res.status_code != 200:
            continue

        soup = BeautifulSoup(res.text, "html.parser")

        # ── Print ALL section headings ──────────────────────────────────────
        print("\n=== ALL SECTIONS ===")
        for sec in soup.find_all("section"):
            h = sec.find("h2") or sec.find("h3")
            has_table = sec.find("table") is not None
            print(f"  heading: '{h.text.strip() if h else None}'  |  has_table: {has_table}")

        # ── Find balance sheet table ────────────────────────────────────────
        bal_table = None
        for sec in soup.find_all("section"):
            h = sec.find("h2") or sec.find("h3")
            if h and "balance" in h.text.lower():
                bal_table = sec.find("table")
                break

        if bal_table is None:
            print("\n!!! NO BALANCE SHEET TABLE FOUND !!!")
            continue

        # ── Print every raw row of the balance sheet table ─────────────────
        print("\n=== BALANCE SHEET TABLE — RAW ROWS ===")
        for i, row in enumerate(bal_table.find_all("tr")):
            cells = [c.get_text(strip=True) for c in row.find_all(["th", "td"])]
            print(f"  Row {i:02d}: {cells}")

        # ── Show what parse_table (old) would produce ───────────────────────
        print("\n=== parse_table RESULT (label → first 3 values) ===")
        rows_dict = {}
        headers = []
        for i, row in enumerate(bal_table.find_all("tr")):
            cells = [c.get_text(strip=True) for c in row.find_all(["th", "td"])]
            if not cells:
                continue
            if i == 0:
                headers = cells
                print(f"  HEADERS: {headers}")
                continue
            rows_dict[cells[0]] = cells[1:]

        for label, vals in rows_dict.items():
            print(f"  '{label}': {vals[:3]}")

        # ── Specifically look for equity-related rows ───────────────────────
        print("\n=== EQUITY-RELATED ROW SEARCH ===")
        equity_patterns = [
            "share capital", "equity share", "reserves", "other equity",
            "surplus", "net worth", "total equity", "shareholders",
            "stockholders", "networth"
        ]
        for label, vals in rows_dict.items():
            lc = label.lower()
            if any(p in lc for p in equity_patterns):
                print(f"  MATCH '{label}': {vals[:5]}")

        print("\n=== DONE ===")
        return  # stop after first successful URL

diagnose_screener("RELIANCE")


Fetching: https://www.screener.in/company/RELIANCE/consolidated/?view=10years
Status: 200  |  Length: 228896

=== ALL SECTIONS ===
  heading: 'None'  |  has_table: False
  heading: 'None'  |  has_table: False
  heading: 'Peer comparison'  |  has_table: False
  heading: 'Quarterly Results'  |  has_table: True
  heading: 'Profit & Loss'  |  has_table: True
  heading: 'Balance Sheet'  |  has_table: True
  heading: 'Cash Flows'  |  has_table: True
  heading: 'Ratios'  |  has_table: True
  heading: 'Insights'  |  has_table: True
  heading: 'Shareholding Pattern'  |  has_table: True
  heading: 'Documents'  |  has_table: False

=== BALANCE SHEET TABLE — RAW ROWS ===
  Row 00: ['', 'Mar 2015', 'Mar 2016', 'Mar 2017', 'Mar 2018', 'Mar 2019', 'Mar 2020', 'Mar 2021', 'Mar 2022', 'Mar 2023', 'Mar 2024', 'Mar 2025', 'Mar 2026']
  Row 01: ['Equity Capital', '2,943', '2,948', '2,959', '5,922', '5,926', '6,339', '6,445', '6,765', '6,766', '6,766', '13,532', '13,532']
  Row 02: ['Reserves', '215,556',

In [33]:
def get_date(row):
    q_map = {'Q1': '06-30', 'Q2': '09-30', 'Q3': '12-31', 'Q4': '03-31', 'FY': '03-31'}
    return pd.to_datetime(f"{row['year']}-{q_map[row['quarter']]}")

final_df['date'] = final_df.apply(get_date, axis=1)

In [34]:
final_df.head()

,symbol,year,quarter,sales,operating_profit,other_income,interest_expense,depreciation,net_income,total_assets,operating_profitability,source,book_value,date
0,360ONE,2018,FY,None,1053.0,None,559.0,14.0,380.0,9567.0,0.265164,screener_html,1863.0,2018-03-31
1,360ONE,2019,FY,None,990.0,None,431.0,22.0,375.0,9763.0,0.192030,screener_html,2911.0,2019-03-31
2,360ONE,2020,FY,None,823.0,None,502.0,41.0,201.0,13021.0,0.107322,screener_html,2991.0,2020-03-31
3,360ONE,2021,FY,None,939.0,None,414.0,43.0,369.0,8739.0,0.185644,screener_html,2828.0,2021-03-31
4,360ONE,2022,FY,None,1157.0,None,370.0,42.0,578.0,10734.0,0.260251,screener_html,3024.0,2022-03-31


In [35]:
final_df['date'] = pd.to_datetime(final_df['date'])

In [36]:
final_df.reset_index(drop=True, inplace=True)

In [37]:
final_df = final_df.sort_values(['symbol', 'date'])

In [38]:
final_df = final_df.set_index('date')

In [39]:
fund_monthly = (
    final_df
    .groupby('symbol', group_keys=False)
    .resample('M')
    .ffill()
    .reset_index()
)

In [40]:
fund_monthly.head(20)

,date,symbol,year,quarter,sales,operating_profit,other_income,interest_expense,depreciation,net_income,total_assets,operating_profitability,source,book_value
0,2018-03-31,360ONE,2018,FY,None,1053.0,None,559.0,14.0,380.0,9567.0,0.265164,screener_html,1863.0
1,2018-04-30,360ONE,2018,FY,None,1053.0,None,559.0,14.0,380.0,9567.0,0.265164,screener_html,1863.0
2,2018-05-31,360ONE,2018,FY,None,1053.0,None,559.0,14.0,380.0,9567.0,0.265164,screener_html,1863.0
3,2018-06-30,360ONE,2018,FY,None,1053.0,None,559.0,14.0,380.0,9567.0,0.265164,screener_html,1863.0
4,2018-07-31,360ONE,2018,FY,None,1053.0,None,559.0,14.0,380.0,9567.0,0.265164,screener_html,1863.0
5,2018-08-31,360ONE,2018,FY,None,1053.0,None,559.0,14.0,380.0,9567.0,0.265164,screener_html,1863.0
6,2018-09-30,360ONE,2018,FY,None,1053.0,None,559.0,14.0,380.0,9567.0,0.265164,screener_html,1863.0
7,2018-10-31,360ONE,2018,FY,None,1053.0,None,559.0,14.0,380.0,9567.0,0.265164,screener_html,1863.0
8,2018-11-30,360ONE,2018,FY,None,1053.0,None,559.0,14.0,380.0,9567.0,0.265164,screener_html,1863.0
9,2018-12-31,360ONE,2018,FY,None,1053.0,None,559.0,14.0,380.0,9567.0,0.265164,screener_html,1863.0


In [41]:
fund_monthly.describe().transpose()

,count,mean,min,25%,50%,75%,max,std
date,57458,2021-01-13 21:30:13.846635776,2011-03-31 00:00:00,2018-05-31 00:00:00,2021-03-31 00:00:00,2023-09-30 00:00:00,2026-03-31 00:00:00,NaN
year,57458.0,2020.295416,2011.0,2018.0,2021.0,2023.0,2026.0,3.240126
operating_profit,50353.0,3140.407523,-19850.0,287.0,751.0,1979.0,179065.0,9239.786412
interest_expense,57458.0,2509.964339,-14.0,17.0,88.0,522.0,315005.0,11821.370005
depreciation,57458.0,827.899126,-11.0,50.0,142.0,422.0,57688.0,3021.898346
net_income,57458.0,1796.792544,-73878.0,137.0,445.0,1308.0,95754.0,6050.462396
total_assets,57458.0,80951.858227,25.0,3164.0,8722.0,32849.0,8321176.0,354854.086369
operating_profitability,49450.0,1.56087,-19.423423,0.129758,0.217486,0.311131,5460.0,85.045386
book_value,57458.0,14101.084852,-104167.0,1463.0,3580.0,10485.0,904030.0,40894.267453


In [42]:
cols_to_lag = ['operating_profit', 
               'interest_expense', 
               'depreciation', 
               'net_income', 
               'total_assets', 
               'operating_profitability',
               'book_value']

fund_monthly[cols_to_lag] = (
    fund_monthly.groupby('symbol')[cols_to_lag]
    .shift(3)
)

In [43]:
# import sys
# !{sys.executable} -m pip install plotly

In [44]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'browser'   # safest across environments

# Step 1: Count + sort years
year_counts = (
    final_df['year']
    .astype(int)
    .value_counts()
    .sort_index()
    .reset_index()
)

year_counts.columns = ['year', 'count']

# Step 2: Plot
fig = px.bar(
    year_counts,
    x='year',
    y='count',
    title='Number of Data Points per Year',
    labels={'year': 'Year', 'count': 'Data Availability: Number of stocks out of 500'}
)

# Optional styling
fig.update_layout(
    xaxis=dict(dtick=1),   # show every year
    template='plotly_white'
)

fig.show()

In [45]:
# All monetary accounting variables are stored in INR (₹),
# not ₹ crore, after conversion from Screener data.

In [46]:
final_df['operating_profit'] = final_df['operating_profit']*(10**7)
final_df['interest_expense'] = final_df['interest_expense']*(10**7) 
final_df['depreciation'] = final_df['depreciation']*(10**7)
final_df['net_income'] = final_df['net_income']*(10**7)
final_df['total_asset'] = final_df['total_assets']*(10**7)
final_df['book_value'] = final_df['book_value']*(10**7)

In [47]:
final_df[['operating_profit', 
        'interest_expense', 
        'depreciation', 
        'net_income', 
        'total_assets', 
        'operating_profitability',
        'book_value']].describe().transpose()


,count,mean,std,min,25%,50%,75%,max
operating_profit,4531.0,3.359563e+10,9.863955e+10,-1.985000e+11,3.240000e+09,8.120000e+09,2.104000e+10,1.790650e+12
interest_expense,5174.0,2.681090e+10,1.276065e+11,-1.400000e+08,1.900000e+08,9.700000e+08,5.427500e+09,3.150050e+12
depreciation,5174.0,8.766832e+09,3.198390e+10,-1.100000e+08,5.200000e+08,1.540000e+09,4.560000e+09,5.768800e+11
net_income,5174.0,2.006806e+10,6.612982e+10,-7.387800e+11,1.542500e+09,4.795000e+09,1.426000e+10,9.575400e+11
total_assets,5174.0,8.677251e+04,3.819706e+05,2.500000e+01,3.409250e+03,9.462000e+03,3.462925e+04,8.321176e+06
operating_profitability,4458.0,1.460804e+00,8.177519e+01,-1.942342e+01,1.325274e-01,2.174333e-01,3.108672e-01,5.460000e+03
book_value,5174.0,1.522375e+11,4.409618e+11,-1.041670e+12,1.611250e+10,3.964000e+10,1.141700e+11,9.040300e+12


In [48]:
final_df.shape

(5174, 14)

In [49]:
final_df.head()

,symbol,year,quarter,sales,operating_profit,other_income,interest_expense,depreciation,net_income,total_assets,operating_profitability,source,book_value,total_asset
date,,,,,,,,,,,,,,
2018-03-31,360ONE,2018,FY,None,1.053000e+10,None,5.590000e+09,140000000.0,3.800000e+09,9567.0,0.265164,screener_html,1.863000e+10,9.567000e+10
2019-03-31,360ONE,2019,FY,None,9.900000e+09,None,4.310000e+09,220000000.0,3.750000e+09,9763.0,0.192030,screener_html,2.911000e+10,9.763000e+10
2020-03-31,360ONE,2020,FY,None,8.230000e+09,None,5.020000e+09,410000000.0,2.010000e+09,13021.0,0.107322,screener_html,2.991000e+10,1.302100e+11
2021-03-31,360ONE,2021,FY,None,9.390000e+09,None,4.140000e+09,430000000.0,3.690000e+09,8739.0,0.185644,screener_html,2.828000e+10,8.739000e+10
2022-03-31,360ONE,2022,FY,None,1.157000e+10,None,3.700000e+09,420000000.0,5.780000e+09,10734.0,0.260251,screener_html,3.024000e+10,1.073400e+11


In [50]:
final_df.to_csv(PROCESSED_DIR / "final_fin_data.csv")

In [51]:
price_data.head()

,date,"ã¯â»â¿""symbol""",series,prevclose,openprice,highprice,lowprice,lastprice,closeprice,averageprice,...,turnoverinrs,no.oftrades,deliverableqty,%dlyqttotradedqty,symbol,return,year,month,effective_year,turnover
0,2011-01-31,MPHASIS,EQ,658.7,657.4,678.9,637.5,678.0,674.65,655.11,...,2.615327e+08,NaN,243120.0,60.9,MPHASIS,NaN,2011,1,2010,2.693331e+08
3,2011-01-31,NHPC,EQ,24.95,24.5,24.85,24.4,24.55,24.55,24.55,...,1.403002e+08,NaN,2514910.0,44.0,NHPC,NaN,2011,1,2010,1.403066e+08
6,2011-01-31,GRASIM,EQ,"2,302.80","2,300.00","2,317.00","2,260.00","2,295.00",2281.25,"2,285.16",...,1.036844e+08,NaN,23041.0,50.78,GRASIM,NaN,2011,1,2010,1.035072e+08
7,2011-01-31,BALRAMCHIN,EQ,72.55,72.0,74.0,69.0,70.7,70.60,70.47,...,1.166330e+08,NaN,486030.0,29.36,BALRAMCHIN,NaN,2011,1,2010,1.168552e+08
8,2011-01-31,UCOBANK,EQ,105.95,105.0,110.0,103.0,108.7,107.65,107.13,...,4.865995e+08,NaN,1193134.0,26.27,UCOBANK,NaN,2011,1,2010,4.889445e+08


In [52]:
price_data.columns

Index(['date', 'ã¯â»â¿"symbol"', 'series', 'prevclose', 'openprice',
       'highprice', 'lowprice', 'lastprice', 'closeprice', 'averageprice',
       'totaltradedquantity', 'turnoverinrs', 'no.oftrades', 'deliverableqty',
       '%dlyqttotradedqty', 'symbol', 'return', 'year', 'month',
       'effective_year', 'turnover'],
      dtype='object')

In [53]:
price_data.columns = price_data.columns.str.replace('ã¯â»â¿"', '', regex=False)
price_data.columns = price_data.columns.str.strip()

In [54]:
price_data.describe().transpose()

,count,mean,min,25%,50%,75%,max,std
date,45721,2018-12-21 11:43:26.950854144,2011-01-31 00:00:00,2015-01-31 00:00:00,2018-12-31 00:00:00,2022-11-30 00:00:00,2026-09-30 00:00:00,NaN
closeprice,45721.0,762.530692,0.6,118.5,321.2,837.75,32233.4,1480.935312
totaltradedquantity,45721.0,5745785.700007,1.0,276190.0,970722.0,3475863.0,4213707883.0,40811748.624134
turnoverinrs,45721.0,1518196760.893429,45.0,76640790.05,403373264.05,1402548082.31,364811305104.200012,4234777244.415256
no.oftrades,44584.0,39337.004598,1.0,6657.75,20598.0,48357.75,1711821.0,59637.98453
deliverableqty,45721.0,2260570.911682,1.0,126311.0,426141.0,1494359.0,2747107638.0,17055463.685441
return,45476.0,0.014592,-0.951383,-0.053341,0.007054,0.07542,5.422222,0.130656
year,45721.0,2018.439514,2011.0,2015.0,2018.0,2022.0,2026.0,4.525211
month,45721.0,6.43234,1.0,3.0,6.0,9.0,12.0,3.429559
effective_year,45721.0,2017.93176,2010.0,2014.0,2018.0,2022.0,2026.0,4.538682


In [55]:
# =====================================================================
# STEP 7: Derive Factor Inputs
# =====================================================================

In [56]:
fund_monthly.head(50)

,date,symbol,year,quarter,sales,operating_profit,other_income,interest_expense,depreciation,net_income,total_assets,operating_profitability,source,book_value
0,2018-03-31,360ONE,2018,FY,None,NaN,None,NaN,NaN,NaN,NaN,NaN,screener_html,NaN
1,2018-04-30,360ONE,2018,FY,None,NaN,None,NaN,NaN,NaN,NaN,NaN,screener_html,NaN
2,2018-05-31,360ONE,2018,FY,None,NaN,None,NaN,NaN,NaN,NaN,NaN,screener_html,NaN
3,2018-06-30,360ONE,2018,FY,None,1053.0,None,559.0,14.0,380.0,9567.0,0.265164,screener_html,1863.0
4,2018-07-31,360ONE,2018,FY,None,1053.0,None,559.0,14.0,380.0,9567.0,0.265164,screener_html,1863.0
5,2018-08-31,360ONE,2018,FY,None,1053.0,None,559.0,14.0,380.0,9567.0,0.265164,screener_html,1863.0
6,2018-09-30,360ONE,2018,FY,None,1053.0,None,559.0,14.0,380.0,9567.0,0.265164,screener_html,1863.0
7,2018-10-31,360ONE,2018,FY,None,1053.0,None,559.0,14.0,380.0,9567.0,0.265164,screener_html,1863.0
8,2018-11-30,360ONE,2018,FY,None,1053.0,None,559.0,14.0,380.0,9567.0,0.265164,screener_html,1863.0
9,2018-12-31,360ONE,2018,FY,None,1053.0,None,559.0,14.0,380.0,9567.0,0.265164,screener_html,1863.0


In [57]:
# =====================================================================
# 7.1 Profitability
# =====================================================================

In [58]:
#profitability or ROE
fund_monthly['operating_profitability'].describe(percentiles = [0.01, 0.10, 0.25, 0.50, 0.75, 0.90, 0.99])

count    48164.000000
mean         1.596250
std         86.173020
min        -19.423423
1%          -0.431818
10%          0.029340
25%          0.129517
50%          0.217557
75%          0.312135
90%          0.422982
99%          1.019368
max       5460.000000
Name: operating_profitability, dtype: float64

In [59]:
print(len(fund_monthly[fund_monthly['operating_profitability'] < -1]))
print(len(fund_monthly[fund_monthly['operating_profitability'] < -2]))

144
60


In [60]:
extreme_op = (
    fund_monthly[fund_monthly['operating_profitability'].abs() > 2][
        ['date', 'symbol', 'operating_profitability', 'operating_profit', 'interest_expense', 'book_value']
    ].sort_values('operating_profitability', ascending=False)
)

extreme_op.head(60)

,date,symbol,operating_profitability,operating_profit,interest_expense,book_value
37397,2022-01-31,MSUMI,5460.000000,553.0,7.0,0.1
37390,2021-06-30,MSUMI,5460.000000,553.0,7.0,0.1
37400,2022-04-30,MSUMI,5460.000000,553.0,7.0,0.1
37399,2022-03-31,MSUMI,5460.000000,553.0,7.0,0.1
37398,2022-02-28,MSUMI,5460.000000,553.0,7.0,0.1
37396,2021-12-31,MSUMI,5460.000000,553.0,7.0,0.1
37395,2021-11-30,MSUMI,5460.000000,553.0,7.0,0.1
37394,2021-10-31,MSUMI,5460.000000,553.0,7.0,0.1
37393,2021-09-30,MSUMI,5460.000000,553.0,7.0,0.1
37392,2021-08-31,MSUMI,5460.000000,553.0,7.0,0.1


In [61]:
fund_monthly.loc[
    fund_monthly['operating_profitability'].abs() > 2,
    'book_value'
].describe()

count     204.000000
mean      620.772353
std       615.425317
min         0.100000
25%       114.000000
50%       457.030000
75%       928.000000
max      1996.000000
Name: book_value, dtype: float64

In [62]:
fund_monthly[
        ['date', 'symbol', 'operating_profitability', 'operating_profit', 'interest_expense', 'book_value']
].sort_values('operating_profitability', ascending=False).nunique()

date                        181
symbol                      494
operating_profitability    4005
operating_profit           2496
interest_expense           1544
book_value                 3963
dtype: int64

In [63]:
june = fund_monthly[fund_monthly['date'].dt.month == 6].copy()

june['operating_profitability'].describe(
    percentiles=[0.01, 0.05, 0.10, 0.25,
                 0.50, 0.75, 0.90, 0.95, 0.99]
)

count    4085.000000
mean        1.572528
std        85.427077
min       -19.423423
1%         -0.412245
5%         -0.044087
10%         0.030309
25%         0.129733
50%         0.217529
75%         0.311475
90%         0.422641
95%         0.527598
99%         1.020488
max      5460.000000
Name: operating_profitability, dtype: float64

In [64]:
fund_monthly['book_value'].describe(percentiles=[0.01, 0.10, 0.25, 0.50, 0.75, 0.90, 0.99])

count     55982.000000
mean      13864.205162
std       40152.905517
min     -104167.000000
1%         -796.750000
10%         529.000000
25%        1424.000000
50%        3519.000000
75%       10345.000000
90%       33576.000000
99%      158180.370000
max      843200.000000
Name: book_value, dtype: float64

In [65]:
fund_monthly = fund_monthly[
    fund_monthly['book_value'] > 0
].copy()

In [66]:
len(fund_monthly)

55086

In [67]:
# =====================================================================
# 7.2 CMA (Investment)
# =====================================================================

In [68]:
fund_annual = (
    fund_monthly
    .sort_values(['symbol', 'year'])
    .drop_duplicates(['symbol', 'year'], keep='last')
    .copy()
)

fund_annual['asset_growth'] = (
    fund_annual.groupby('symbol')['total_assets']
    .pct_change()
)

In [69]:
rf_monthly.head()

,Rf_monthly
DATE,
2011-11-30,0.007315
2011-12-31,0.007290
2012-01-31,0.007342
2012-02-29,0.007110
2012-03-31,0.007692


In [70]:
rf_monthly = rf_monthly.reset_index()

In [71]:
rf_monthly.rename(columns={'DATE': 'date'}, inplace=True)

In [72]:
price_data['date'] = price_data['date'] + pd.offsets.MonthEnd(0)
fund_monthly['date'] = fund_monthly['date'] + pd.offsets.MonthEnd(0)

In [73]:
final = price_data.merge(fund_monthly, on=['symbol', 'date'], how='left')
final = final.merge(rf_monthly, on='date', how='left')

In [74]:
# Merge Annual Asset Growth into Monthly Panel

# 1. Prepare annual asset-growth characteristic

# fund_annual contains one observation per stock and
# fiscal/accounting year.
#
# asset_growth(t) =
#     Total Assets(t) / Total Assets(t-1) - 1
#
# For July formation in year T, we use:
#
#     FY(T-1) asset growth
#
# Therefore:
#
#     fiscal_year + 1 = formation_year

asset_growth_formation = fund_annual[
    [
        'symbol',
        'year',
        'asset_growth'
    ]
].copy()

asset_growth_formation = asset_growth_formation.rename(
    columns={
        'year': 'fiscal_year'
    }
)

asset_growth_formation['formation_year'] = (
    asset_growth_formation['fiscal_year']
)

asset_growth_formation = asset_growth_formation[
    [
        'symbol',
        'formation_year',
        'asset_growth'
    ]
].copy()


# 2. Remove any existing asset_growth from final

# final already received columns from fund_monthly earlier.
# If asset_growth exists there, we do NOT want to keep it,
# because that may be the old/monthly version.
#
# We want only the correctly constructed annual
# FY(t-1) investment characteristic.

asset_growth_cols = [
    col for col in final.columns
    if col.startswith('asset_growth')
]

if asset_growth_cols:
    final = final.drop(columns=asset_growth_cols)


# 3. Create portfolio formation year

# July 2020 -> formation year 2020
# August 2020 -> formation year 2020
# ...
# June 2021 -> formation year 2020
#
# July 2021 -> formation year 2021

final['formation_year'] = np.where(
    final['date'].dt.month >= 7,
    final['date'].dt.year,
    final['date'].dt.year - 1
)


# 4. Merge annual investment characteristic
final = final.merge(
    asset_growth_formation,
    on=['symbol', 'formation_year'],
    how='left'
)


# 5. Validation
print(
    "Asset growth non-null:",
    final['asset_growth'].notna().sum()
)

print(
    "Asset growth columns:",
    [
        col for col in final.columns
        if 'asset_growth' in col
    ]
)

Asset growth non-null: 28684
Asset growth columns: ['asset_growth']


In [75]:
final[['year_x', 'year_y']].head()

,year_x,year_y
0,2011,NaN
1,2011,NaN
2,2011,NaN
3,2011,NaN
4,2011,NaN


In [76]:
final.drop('year_y', axis=1, inplace=True)

In [77]:
final = final.rename(columns = {'year_x': 'year'})

In [78]:
print(
    final[
        [
            'date',
            'symbol',
            'year',
            'effective_year',
            'formation_year',
            'asset_growth'
        ]
    ]
    .dropna(subset=['asset_growth'])
    .head(20)
)

           date     symbol  year  effective_year  formation_year  asset_growth
4207 2012-07-31        ABB  2012            2012            2012      0.029277
4308 2012-07-31  TATAELXSI  2012            2012            2012      0.109272
4410 2012-08-31  TATAELXSI  2012            2012            2012      0.109272
4489 2012-08-31        ABB  2012            2012            2012      0.029277
4727 2012-09-30        ABB  2012            2012            2012      0.029277
4738 2012-09-30  TATAELXSI  2012            2012            2012      0.109272
4991 2012-10-31        ABB  2012            2012            2012      0.029277
5054 2012-10-31  TATAELXSI  2012            2012            2012      0.109272
5123 2012-11-30  TATAELXSI  2012            2012            2012      0.109272
5324 2012-11-30        ABB  2012            2012            2012      0.029277
5439 2012-12-31  TATAELXSI  2012            2012            2012      0.109272
5551 2012-12-31        ABB  2012            2012    

In [79]:
# ==========================================================
# Merge Book Value (FINAL STEP)
# ==========================================================

# Rename the duplicate column created by the fund_monthly merge if present
if 'book_value_x' in final.columns:
    final = final.drop(columns=[c for c in final.columns if c == 'book_value_y'])
    final = final.rename(columns={'book_value_x': 'book_value'})

print("Book value after merge:", final['book_value'].notna().sum())
print("Final shape:", final.shape)
print(final[['symbol', 'date', 'book_value']].dropna(subset=['book_value']).head(10))

Book value after merge: 30447
Final shape: (45721, 35)
         symbol       date  book_value
1296  TATAELXSI 2011-06-30       182.0
1325        ABB 2011-06-30      2534.0
1472        ABB 2011-07-31      2534.0
1567  TATAELXSI 2011-07-31       182.0
1686  TATAELXSI 2011-08-31       182.0
1759        ABB 2011-08-31      2534.0
1864        ABB 2011-09-30      2534.0
2031  TATAELXSI 2011-09-30       182.0
2098  TATAELXSI 2011-10-31       182.0
2108        ABB 2011-10-31      2534.0


In [80]:
final.head()

,date,"symbol""",series,prevclose,openprice,highprice,lowprice,lastprice,closeprice,averageprice,...,interest_expense,depreciation,net_income,total_assets,operating_profitability,source,book_value,Rf_monthly,formation_year,asset_growth
0,2011-01-31,MPHASIS,EQ,658.7,657.4,678.9,637.5,678.0,674.65,655.11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2010,NaN
1,2011-01-31,NHPC,EQ,24.95,24.5,24.85,24.4,24.55,24.55,24.55,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2010,NaN
2,2011-01-31,GRASIM,EQ,"2,302.80","2,300.00","2,317.00","2,260.00","2,295.00",2281.25,"2,285.16",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2010,NaN
3,2011-01-31,BALRAMCHIN,EQ,72.55,72.0,74.0,69.0,70.7,70.60,70.47,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2010,NaN
4,2011-01-31,UCOBANK,EQ,105.95,105.0,110.0,103.0,108.7,107.65,107.13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2010,NaN


In [81]:
final.shape

(45721, 35)

In [82]:
final.isnull().sum()

date                           0
symbol"                        0
series                         0
prevclose                      0
openprice                      0
highprice                      0
lowprice                       0
lastprice                      0
closeprice                     0
averageprice                   0
totaltradedquantity            0
turnoverinrs                   0
no.oftrades                 1137
deliverableqty                 0
%dlyqttotradedqty              0
symbol                         0
return                       245
year                           0
month                          0
effective_year                 0
turnover                       0
quarter                    15274
sales                      45721
operating_profit           19912
other_income               45721
interest_expense           15274
depreciation               15274
net_income                 15274
total_assets               15274
operating_profitability    19912
source    

In [83]:
# =====================================================================
# 7.3 Market Cap
# =====================================================================

Market Cap = Price × Shares Outstanding

In [84]:
shares_history = {}

for sym in price_data['symbol'].unique():
    try:
        ticker = yf.Ticker(f"{sym}.NS")
        sh = ticker.get_shares_full(start="2011-01-01")
        if sh is not None and not sh.empty:
            sh = sh.resample("M").last().ffill()
            shares_history[sym] = sh
    except:
        shares_history[sym] = None

# Build a long-format DataFrame instead of a static map
shares_frames = []
for sym, sh in shares_history.items():
    if sh is not None:
        temp = sh.reset_index()
        temp.columns = ['date', 'shares_outstanding']
        temp['date'] = pd.to_datetime(temp['date']).dt.tz_localize(None)  # strip timezone
        temp['date'] = temp['date'] + pd.offsets.MonthEnd(0)
        temp['symbol'] = sym
        shares_frames.append(temp)

shares_df = pd.concat(shares_frames, ignore_index=True)

# Merge by both symbol AND date — not a static map
final = final.merge(shares_df, on=['symbol', 'date'], how='left')
final['market_cap'] = final['closeprice'] * final['shares_outstanding']
final['market_cap'] = pd.to_numeric(final['market_cap'], errors='coerce')

# Debug
print("Non-null shares:", final['shares_outstanding'].notna().sum())
print("Non-null market cap:", final['market_cap'].notna().sum())


Non-null shares: 25518
Non-null market cap: 25518


In [85]:
# ==========================================================
# SAVE MARKET CAP PANEL
# ==========================================================

market_cap_df = final[
    ['date', 'symbol', 'market_cap']
].copy()

market_cap_df.to_csv(
    PROCESSED_DIR / "market_cap_monthly.csv",
    index=False
)

print("Saved: market_cap_monthly.csv")

Saved: market_cap_monthly.csv


In [86]:
print("Total stocks:", len(shares_history))

print(
    "Stocks with historical shares:",
    sum(v is not None for v in shares_history.values())
)

print(
    "Stocks missing shares:",
    sum(v is None for v in shares_history.values())
)

print(
    "Rows with market cap:",
    final['market_cap'].notna().sum()
)

Total stocks: 245
Stocks with historical shares: 245
Stocks missing shares: 0
Rows with market cap: 25518


In [87]:
coverage = (
    final.groupby('date')['market_cap']
    .apply(lambda x: x.notna().mean())
)

print(coverage.describe())

count    189.000000
mean       0.553175
std        0.420067
min        0.000000
25%        0.043103
50%        0.865306
75%        0.905350
max        1.000000
Name: market_cap, dtype: float64


In [88]:
# =====================================================================
# 7.4 Book-to-Market
# =====================================================================

* Book Value = Shareholders' Equity (Net Worth)
* B/M = Book Value / Market Cap

In [89]:
final.head()

,date,"symbol""",series,prevclose,openprice,highprice,lowprice,lastprice,closeprice,averageprice,...,net_income,total_assets,operating_profitability,source,book_value,Rf_monthly,formation_year,asset_growth,shares_outstanding,market_cap
0,2011-01-31,MPHASIS,EQ,658.7,657.4,678.9,637.5,678.0,674.65,655.11,...,NaN,NaN,NaN,NaN,NaN,NaN,2010,NaN,NaN,NaN
1,2011-01-31,NHPC,EQ,24.95,24.5,24.85,24.4,24.55,24.55,24.55,...,NaN,NaN,NaN,NaN,NaN,NaN,2010,NaN,NaN,NaN
2,2011-01-31,GRASIM,EQ,"2,302.80","2,300.00","2,317.00","2,260.00","2,295.00",2281.25,"2,285.16",...,NaN,NaN,NaN,NaN,NaN,NaN,2010,NaN,NaN,NaN
3,2011-01-31,BALRAMCHIN,EQ,72.55,72.0,74.0,69.0,70.7,70.60,70.47,...,NaN,NaN,NaN,NaN,NaN,NaN,2010,NaN,NaN,NaN
4,2011-01-31,UCOBANK,EQ,105.95,105.0,110.0,103.0,108.7,107.65,107.13,...,NaN,NaN,NaN,NaN,NaN,NaN,2010,NaN,NaN,NaN


In [90]:
final.columns

Index(['date', 'symbol"', 'series', 'prevclose', 'openprice', 'highprice',
       'lowprice', 'lastprice', 'closeprice', 'averageprice',
       'totaltradedquantity', 'turnoverinrs', 'no.oftrades', 'deliverableqty',
       '%dlyqttotradedqty', 'symbol', 'return', 'year', 'month',
       'effective_year', 'turnover', 'quarter', 'sales', 'operating_profit',
       'other_income', 'interest_expense', 'depreciation', 'net_income',
       'total_assets', 'operating_profitability', 'source', 'book_value',
       'Rf_monthly', 'formation_year', 'asset_growth', 'shares_outstanding',
       'market_cap'],
      dtype='object')

In [91]:
final[['effective_year', 'year']].isnull().sum()

effective_year    0
year              0
dtype: int64

In [92]:
len(final)

45721

In [93]:
# 1: December market cap
# dec_me -> dec market equity

dec_me = final[final['month'] == 12][
    ['symbol', 'year', 'market_cap']
].copy()

dec_me = dec_me.rename(
    columns={
        'year': 'dec_year',
        'market_cap': 'dec_market_cap'
    }
)


In [94]:
# 2: Previous fiscal-year book equity

# Book equity from fiscal year t-1.
# Take the latest available accounting observation
# for each stock within each fiscal year.

book_equity_annual = (
    final[
        final['book_value'].notna()
    ]
    .sort_values(['symbol', 'year', 'month'])
    .groupby(['symbol', 'year'], as_index=False)
    .last()
    [['symbol', 'year', 'book_value']]
)

book_equity_annual = book_equity_annual.rename(columns={
    'year': 'fiscal_year',
    'book_value': 'book_equity'
    }
)


In [95]:
# 3: Create formation year

# For July formation in year t:
#   Book Equity  -> FY t-1
#   December ME  -> December t-1
#
# Therefore:
#   formation_year = fiscal_year + 1

book_equity_annual['formation_year'] = (
    book_equity_annual['fiscal_year']
)

dec_me['formation_year'] = (
    dec_me['dec_year'] + 1
)


In [96]:
# 4: Merge book equity with December market cap

bm_formation = book_equity_annual.merge(
    dec_me[
        [
            'symbol',
            'formation_year',
            'dec_market_cap'
        ]
    ],
    on=['symbol', 'formation_year'],
    how='inner'
)


In [97]:
# 5: Calculate B/M

bm_formation['book_equity'] = bm_formation['book_equity']*(10**7)

bm_formation['bm'] = (
    bm_formation['book_equity']
    / bm_formation['dec_market_cap']
)

# Keep only economically meaningful observations
bm_formation = bm_formation[
    (bm_formation['book_equity'] > 0) &
    (bm_formation['dec_market_cap'] > 0) &
    np.isfinite(bm_formation['bm']) &
    (bm_formation['bm'] > 0)
].copy()


In [98]:
bm_formation.head()

,symbol,fiscal_year,book_equity,formation_year,dec_market_cap,bm
3,AARTIIND,2018,1.579000e+10,2018,9.354550e+10,0.168795
4,AARTIIND,2019,2.630000e+10,2019,1.175875e+11,0.223663
5,AARTIIND,2020,2.979000e+10,2020,1.446934e+11,0.205884
6,AARTIIND,2021,3.503000e+10,2021,2.151886e+11,0.162787
7,AARTIIND,2022,4.516000e+10,2022,3.683205e+11,0.122611


In [99]:
final.columns

Index(['date', 'symbol"', 'series', 'prevclose', 'openprice', 'highprice',
       'lowprice', 'lastprice', 'closeprice', 'averageprice',
       'totaltradedquantity', 'turnoverinrs', 'no.oftrades', 'deliverableqty',
       '%dlyqttotradedqty', 'symbol', 'return', 'year', 'month',
       'effective_year', 'turnover', 'quarter', 'sales', 'operating_profit',
       'other_income', 'interest_expense', 'depreciation', 'net_income',
       'total_assets', 'operating_profitability', 'source', 'book_value',
       'Rf_monthly', 'formation_year', 'asset_growth', 'shares_outstanding',
       'market_cap'],
      dtype='object')

In [100]:
final[['date', 'prevclose', 'openprice', 'highprice', 'lowprice', 'lastprice', 'closeprice', 'symbol', 'return', 'year', 
       'month', 'effective_year']].head()

,date,prevclose,openprice,highprice,lowprice,lastprice,closeprice,symbol,return,year,month,effective_year
0,2011-01-31,658.7,657.4,678.9,637.5,678.0,674.65,MPHASIS,NaN,2011,1,2010
1,2011-01-31,24.95,24.5,24.85,24.4,24.55,24.55,NHPC,NaN,2011,1,2010
2,2011-01-31,"2,302.80","2,300.00","2,317.00","2,260.00","2,295.00",2281.25,GRASIM,NaN,2011,1,2010
3,2011-01-31,72.55,72.0,74.0,69.0,70.7,70.60,BALRAMCHIN,NaN,2011,1,2010
4,2011-01-31,105.95,105.0,110.0,103.0,108.7,107.65,UCOBANK,NaN,2011,1,2010


In [101]:
# 6: Create formation year and merge B/M

# FF portfolio formation year:
#
# July 2020 -> June 2021 = formation year 2020
# July 2021 -> June 2022 = formation year 2021
#
# Therefore:
# July-Dec -> current calendar year
# Jan-Jun  -> previous calendar year

final['formation_year'] = np.where(
    final['month'] >= 7,
    final['year'],
    final['year'] - 1
)

# Merge the formation-year B/M back to the monthly panel
final = final.merge(
    bm_formation[
        [
            'symbol',
            'formation_year',
            'bm'
        ]
    ],
    on=['symbol', 'formation_year'],
    how='left'
)

print("B/M non-null:", final['bm'].notna().sum())

print(
    final[
        [
            'date',
            'symbol',
            'year',
            'effective_year',
            'formation_year',
            'bm'
        ]
    ]
    .dropna(subset=['bm'])
    .head(20)
)

B/M non-null: 22269
            date      symbol  year  effective_year  formation_year        bm
10019 2014-07-31         ACC  2014            2014            2014  0.394788
10229 2014-08-31         ACC  2014            2014            2014  0.394788
10450 2014-09-30         ACC  2014            2014            2014  0.394788
10671 2014-10-31         ACC  2014            2014            2014  0.394788
10979 2014-11-30         ACC  2014            2014            2014  0.394788
11201 2014-12-31         ACC  2014            2014            2014  0.394788
11402 2015-01-31         ACC  2015            2014            2014  0.394788
11602 2015-02-28         ACC  2015            2014            2014  0.394788
11922 2015-03-31         ACC  2015            2014            2014  0.394788
12096 2015-04-30         ACC  2015            2014            2014  0.394788
12364 2015-05-31         ACC  2015            2014            2014  0.394788
12678 2015-06-30         ACC  2015            2014      

In [102]:
print("Final shape before filters:", final.shape)
print("Market cap non-null:", final['market_cap'].notna().sum())
print("Book value non-null:", final['book_value'].notna().sum())
print("BM non-null:", final['bm'].notna().sum())

Final shape before filters: (45721, 38)
Market cap non-null: 25518
Book value non-null: 30447
BM non-null: 22269


In [103]:
final = final[
    final['market_cap'] > 0
].copy()

In [104]:
final.columns

Index(['date', 'symbol"', 'series', 'prevclose', 'openprice', 'highprice',
       'lowprice', 'lastprice', 'closeprice', 'averageprice',
       'totaltradedquantity', 'turnoverinrs', 'no.oftrades', 'deliverableqty',
       '%dlyqttotradedqty', 'symbol', 'return', 'year', 'month',
       'effective_year', 'turnover', 'quarter', 'sales', 'operating_profit',
       'other_income', 'interest_expense', 'depreciation', 'net_income',
       'total_assets', 'operating_profitability', 'source', 'book_value',
       'Rf_monthly', 'formation_year', 'asset_growth', 'shares_outstanding',
       'market_cap', 'bm'],
      dtype='object')

In [105]:
# ==========================================================
# SAVE CHARACTERISTICS PANEL
# ==========================================================

characteristics = final[
    [
        'date',
        'symbol',
        'market_cap',
        'bm',
        'operating_profitability',
        'asset_growth'
    ]
].copy()

characteristics.to_csv(
    PROCESSED_DIR / "characteristics_monthly.csv",
    index=False
)

print("Saved: characteristics_monthly.csv")

Saved: characteristics_monthly.csv


In [106]:
final.shape

(25518, 38)

In [107]:
print(final.columns)

Index(['date', 'symbol"', 'series', 'prevclose', 'openprice', 'highprice',
       'lowprice', 'lastprice', 'closeprice', 'averageprice',
       'totaltradedquantity', 'turnoverinrs', 'no.oftrades', 'deliverableqty',
       '%dlyqttotradedqty', 'symbol', 'return', 'year', 'month',
       'effective_year', 'turnover', 'quarter', 'sales', 'operating_profit',
       'other_income', 'interest_expense', 'depreciation', 'net_income',
       'total_assets', 'operating_profitability', 'source', 'book_value',
       'Rf_monthly', 'formation_year', 'asset_growth', 'shares_outstanding',
       'market_cap', 'bm'],
      dtype='object')


In [108]:
# ==========================================================
# FINAL COLUMN CLEANUP
# ==========================================================

# Remove corrupted NSE column
final = final.drop(columns=['symbol"'], errors='ignore')

# Remove duplicate accounting year
final = final.drop(columns=['year_y'], errors='ignore')

# Rename market year
final = final.rename(columns={'year_x': 'year'})

# Optional: reorder important columns
important_cols = [
    'date',
    'symbol',
    'year',
    'month',
    'return',
    'Rf_monthly',
    'market_cap',
    'bm',
    'operating_profitability',
    'asset_growth'
]

remaining_cols = [
    c for c in final.columns
    if c not in important_cols
]

final = final[important_cols + remaining_cols]

In [109]:
# ==========================================================
# SAVE ELIGIBLE STOCK UNIVERSE
# ==========================================================

eligible = final[
    ['date', 'symbol']
].drop_duplicates()

eligible.to_csv(
    PROCESSED_DIR / "eligible_universe_phase1_end.csv",
    index=False
)

print("Saved: eligible_universe.csv")

Saved: eligible_universe.csv


In [110]:
# ==========================================================
# SAVE MASTER MONTHLY PANEL
# ==========================================================

PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

final.to_csv(
    PROCESSED_DIR / "final_panel_phase1_end.csv",
    index=False
)

print("Saved: final_panel.csv")

Saved: final_panel.csv


---

## Phase 1 Summary — Data Ingestion & Cleaning ✓

### What was achieved

| Dataset | Coverage | Stored at |
|---|---|---|
| Price + volume (NSE EQ series) | 2011–present, ~470 stocks, monthly | `data/raw/*.csv` |
| Fundamentals: net income, total assets, book value | 2011–present, 476 stocks, annual (Screener) | `data/fin_data_processed/final_fin_data.csv` |
| Risk-free rate (91-day T-bill, FRED) | 2011–present, monthly compounded | `data/rf_india.csv` |

**Factor inputs derived:**
- `profitability` = Net Income / Total Assets → feeds **RMW**
- `asset_growth` = % change in Total Assets → feeds **CMA**
- `book_value` (lagged 3 months, annual → monthly expanded) → feeds **HML**
- `market_cap` = Price × Shares Outstanding → feeds **SMB** and **HML**
- `bm` = Book Value / Market Cap → feeds **HML**
- `return` and `Rf_monthly` → feed **MKT**

All fundamentals are lagged 3 months to prevent lookahead bias. Stocks with fewer than 70% monthly observations or median volume below 100k shares were dropped.

---

### How this feeds Phase 2 — Factor Construction

In Phase 2 we use `final` (the merged monthly panel) to construct all five Fama–French factors:

- **MKT** — value-weighted market return minus `Rf_monthly`, computed across all stocks each month
- **SMB** — monthly return spread between bottom and top tercile of `market_cap`, size-sorted portfolios
- **HML** — monthly return spread between top and bottom tercile of `bm`, annually rebalanced each July
- **RMW** — monthly return spread between top and bottom tercile of `profitability`
- **CMA** — monthly return spread between bottom and top tercile of `asset_growth` (low investment minus high)

Portfolio sorts follow the standard FF methodology: independent 2×3 sorts on size and each characteristic, rebalanced annually in July using prior fiscal year-end fundamentals (already lagged in Phase 1).